In [1]:
import jax
import jax.numpy as jnp
from flax import nnx


%load_ext autoreload
%autoreload 2
from resnext.model_jax import ResNeXt

In [2]:
linear = nnx.Linear(in_features=4, out_features=2, rngs=nnx.Rngs(42))

# Flax created a `Param` wrapper over the actual `jax.Array` parameter to track metadata
print(type(linear.kernel))        # flax.nnx.Param
print(type(linear.kernel.value))  # jax.Array

<class 'flax.nnx.variablelib.Param'>
<class 'jaxlib._jax.ArrayImpl'>


In [3]:
# Flatten allows you to see all the content inside a pytree
arrays, treedef = jax.tree.flatten_with_path(linear)
assert len(arrays) > 1
for kp, value in arrays:
    print(f'linear{jax.tree_util.keystr(kp)}: {value}')
print(f'{treedef = }')

# Unflatten brings the pytree back intact
linear2 = jax.tree.unflatten(treedef, [value for _, value in arrays])

linear.bias.value: [0. 0.]
linear.kernel.value: [[ 0.04119058 -0.26290742]
 [ 0.6772457   0.28073984]
 [ 0.16276604  0.1681385 ]
 [ 0.31097504 -0.43336973]]
treedef = PyTreeDef(CustomNode(Linear[(('_pytree__state', 'bias', 'kernel'), (('_pytree__nodes', {'_pytree__state': True, 'kernel': True, 'bias': True, 'in_features': False, 'out_features': False, 'use_bias': False, 'dtype': False, 'param_dtype': False, 'precision': False, 'kernel_init': False, 'bias_init': False, 'dot_general': False, 'promote_dtype': False, 'preferred_element_type': False, '_pytree__nodes': False}), ('bias_init', <function zeros at 0x796ef1001580>), ('dot_general', <function dot_general at 0x796ef1aa4c20>), ('dtype', None), ('in_features', 4), ('kernel_init', <function variance_scaling.<locals>.init at 0x796ef0463380>), ('out_features', 2), ('param_dtype', <class 'jax.numpy.float32'>), ('precision', None), ('preferred_element_type', None), ('promote_dtype', <function promote_dtype at 0x796ef0463600>), ('use_bias'

In [4]:
rngs = nnx.Rngs(0)
resnext = ResNeXt(rngs=rngs)

In [5]:
# e.g., 1 image, 224×224, 3 channels
x = jax.random.normal(jax.random.key(1), (1, 224, 224, 3))
y = resnext(x)
print("Input shape:", x.shape)
print("Output shape:", y.shape)

Input shape: (1, 224, 224, 3)
Output shape: (1, 1000)
